[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C36_GPU_Kernels_Course/04_reductions_softmax/04_reductions_softmax.ipynb)

# 04 · 并行规约与融合 softmax（用 numpy 模拟）

目标：把 **树形规约**、**warp-shuffle 规约**、**数值稳定 softmax**，以及最关键的 **online softmax**（一遍流式 + 分块合并）从零用 numpy 写出来，并与朴素参考 `np.allclose` 对拍。

路线：树形规约 → warp shuffle → 浮点结合律 → 稳定 softmax → **online softmax** → **分块 online softmax**（FlashAttention 的发动机）→ ✏️ 练习 → 📖 答案 → 🧪 真实注意力行胶囊。

> 心智模型：**一轮规约 = 一半线程把另一半加过来 + 一次同步**；**online softmax = 边扫边用校正因子修正**。我们写的是并行*结构*与*递推*，不是性能。

## 1 · 树形规约：O(log n) 深度的求和/求最大

串行求和有一条 O(n) 的依赖链。树形规约用结合律两两配对、每轮砍半，深度降到 **O(log n)**。

下面用「步长减半」模式模拟 GPU 的每一轮（`buf[:stride] += buf[stride:2*stride]`），并与 `np.sum` 对拍。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def tree_reduce_sum(x):
    '''模拟 GPU 树形规约求和：每轮一半「线程」把另一半加过来，O(log n) 轮。'''
    a = np.asarray(x, dtype=float).copy()
    n = len(a)
    size = 1 << (n - 1).bit_length() if n > 1 else 1   # 补到 2 的幂
    buf = np.zeros(size); buf[:n] = a                  # 用单位元 0 补齐
    stride = size // 2
    rounds = 0
    while stride >= 1:
        buf[:stride] = buf[:stride] + buf[stride:2 * stride]   # 半数线程活跃
        stride //= 2; rounds += 1
    return buf[0], rounds

x = rng.standard_normal(1000)
s, rounds = tree_reduce_sum(x)
print(f'n=1000：树形规约 {rounds} 轮（串行需 1000 步），结果={s:.6f}, np.sum={x.sum():.6f}')
assert np.isclose(s, x.sum()), '树形求和应与 np.sum 一致'
assert rounds == 10, '1024 个槽位 -> log2(1024)=10 轮'
print('✅ 树形规约对拍 np.sum 通过；深度 O(log n)')

## 2 · 同一棵树做求最大

把二元算子从「加」换成「取最大」、单位元从 `0` 换成 `-inf`，**完全相同的树结构**就能并行求最大。这说明规约树对任何满足结合律的算子都成立。

In [ ]:
def tree_reduce_max(x):
    a = np.asarray(x, dtype=float).copy()
    n = len(a)
    size = 1 << (n - 1).bit_length() if n > 1 else 1
    buf = np.full(size, -np.inf); buf[:n] = a          # 单位元 = -inf
    stride = size // 2
    while stride >= 1:
        buf[:stride] = np.maximum(buf[:stride], buf[stride:2 * stride])
        stride //= 2
    return buf[0]

x = rng.standard_normal(777)                            # 故意非 2 的幂
m = tree_reduce_max(x)
print(f'n=777：树形求最大={m:.6f}, np.max={x.max():.6f}')
assert np.isclose(m, x.max()), '树形求最大应与 np.max 一致（-inf 补位不影响 max）'
print('✅ 同一棵树换个算子+单位元就求最大；非 2 的幂也正确')

## 3 · warp shuffle 规约：32 lane、5 步、无 shared

warp 内 32 个线程锁步执行，可用 `shfl_down(offset)` 直接交换寄存器，省掉 shared memory 往返与 `__syncthreads()`。

规约只需 5 步：`offset = 16, 8, 4, 2, 1`，每步 `v[t] += v[t+offset]`。我们用长度 32 的数组切片模拟这条 shuffle 链。

In [ ]:
WARP = 32

def warp_reduce_sum(vals):
    '''模拟 warp 内 shuffle-down 规约：5 步折半。vals 长度 <= 32。返回 lane0 持有的总和。'''
    buf = np.zeros(WARP); v = np.asarray(vals, dtype=float)
    buf[:len(v)] = v
    offset = 16
    steps = 0
    while offset >= 1:
        buf[:offset] = buf[:offset] + buf[offset:2 * offset]   # shfl_down(offset)
        offset //= 2; steps += 1
    return buf[0], steps

vals = rng.standard_normal(32)
s, steps = warp_reduce_sum(vals)
print(f'warp 规约 {steps} 步（16,8,4,2,1），结果={s:.6f}, 真值={vals.sum():.6f}')
assert np.isclose(s, vals.sum())
assert steps == 5, 'log2(32)=5 步'
# 不满 32 的 warp（用 0 补位）也正确
s2, _ = warp_reduce_sum(rng.standard_normal(20))
print('✅ warp shuffle 规约对拍通过；5 步、无需 shared/barrier')

### block 规约 = 两级：warp 内 shuffle + warp 间 shared

完整的 block 规约把上面两件事拼起来：每个 warp 先 shuffle 成 1 个数 → 写进一小块 shared → 第一个 warp 再 shuffle 这些 warp 结果。下面模拟这个两级结构。

In [ ]:
def block_reduce_sum(x, block_size=256):
    '''两级 block 规约：① 每个 warp 内 shuffle 求和；② warp 结果再 warp-规约。'''
    x = np.asarray(x, dtype=float)
    assert len(x) == block_size
    n_warps = block_size // WARP
    warp_partials = np.zeros(n_warps)
    for w in range(n_warps):                              # ① 每个 warp 独立规约
        seg = x[w * WARP:(w + 1) * WARP]
        warp_partials[w], _ = warp_reduce_sum(seg)        # 写入 shared 的位置
    total, _ = warp_reduce_sum(warp_partials)             # ② 第一个 warp 规约 warp 结果
    return total

x = rng.standard_normal(256)
tot = block_reduce_sum(x, 256)
print(f'block(256) 两级规约={tot:.6f}, np.sum={x.sum():.6f}')
assert np.isclose(tot, x.sum())
print('✅ warp 内用 shuffle、warp 间用 shared 的两级规约对拍通过')

## 4 · 浮点不结合：树形 vs 串行的微小差异

浮点加法不满足结合律——不同求和顺序会有不同的舍入。所以**并行规约和串行规约结果可能差最后几位**，这是正常现象，不是 bug。对拍时要用容差而非 `==`。

In [ ]:
# 经典反例：大数吃掉小数，顺序不同结果不同
print('(1e16 + 1) - 1e16 =', (1e16 + 1) - 1e16, '  <- 1 被舍入吞掉')
print('1e16 + (1 - 1e16) =', 1e16 + (1 - 1e16), '  <- 顺序变了，1 保住了')

# 在一个病态数组上比较串行 vs 树形求和
bad = np.array([1e16, 1.0, -1e16, 1.0] * 1000)
serial = 0.0
for v in bad:                       # 串行累加：acc 一直很大，小数被淹
    serial += v
tree, _ = tree_reduce_sum(bad)      # 树形：相近量级先加，误差小
true_val = 2000.0                   # 2000 个 +1.0（±1e16 两两抵消）
print(f'\n真值={true_val}  串行={serial}  树形={tree}')
print(f'串行误差={abs(serial-true_val):.1f}  树形误差={abs(tree-true_val):.1f}')
assert abs(tree - true_val) <= abs(serial - true_val), '树形(成对)求和通常更精确'
assert not np.isclose(serial, tree), '两种顺序结果不同 -> 浮点不结合'
print('✅ 证实：浮点不结合；树形(成对)求和往往比串行更精确 -> 对拍用 np.allclose 而非 ==')

## 5 · 数值稳定 softmax 与它的三遍代价

朴素 `exp(x)/exp(x).sum()` 在大 logit 下溢出成 `nan`。减去每行最大值再 exp（分子分母同乘 `e^{-m}`，值不变）即稳定。

数一数：求 max（规约一）+ 求 sum（规约二）+ 写回，**三遍扫描整行**。

In [ ]:
def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum()

def softmax_stable(x):
    m = x.max()                     # pass 1: 求最大
    e = np.exp(x - m)               # pass 2: 求 exp 与其和
    return e / e.sum()              # pass 3: 归一化写回

x = rng.standard_normal(8)
assert np.allclose(softmax_naive(x), softmax_stable(x), atol=1e-12), '小值时两者一致'

big = np.array([1000.0, 1001.0, 1002.0])
out_naive = softmax_naive(big)
out_stable = softmax_stable(big)
print('朴素(溢出):', out_naive, ' 含 nan =', np.isnan(out_naive).any())
print('稳定(正确):', np.round(out_stable, 4), ' 和 =', round(out_stable.sum(), 6))
assert np.isnan(out_naive).any() and not np.isnan(out_stable).any()
assert np.isclose(out_stable.sum(), 1.0)
print('✅ 稳定 softmax 正确；但它要三遍扫描整行 —— 下一节压成一遍')

## 6 · online softmax：一遍流式归一化 ⭐

本模块的核心。维护运行最大值 `m` 与运行归一化和 `l`，每来一个值就更新；遇到更大的值时用**校正因子** `exp(m - m_new)` 把旧的 `l` 回缩到新基准。

$$m_{new}=\max(m,x_i),\quad l \leftarrow l\cdot e^{m-m_{new}} + e^{x_i-m_{new}}$$

扫完后 `softmax(x_i)=e^{x_i-m}/l`。可证它与三遍 softmax **数学完全等价**。

In [ ]:
def online_softmax(x):
    '''一遍流式 softmax（Milakov & Gimelshein 2018）。'''
    m = -np.inf      # 运行最大值
    l = 0.0          # 运行归一化和 Σ e^{xj - m}（相对当前 m）
    for xi in x:
        m_new = max(m, xi)
        # 校正旧和(乘 e^{m-m_new}) + 加入新项。第一步 m=-inf: e^{-inf}=0, l=0*0+1=1
        l = l * np.exp(m - m_new) + np.exp(xi - m_new)
        m = m_new
    return np.exp(x - m) / l

# 对拍随机行
x = rng.standard_normal(64)
assert np.allclose(online_softmax(x), softmax_stable(x), atol=1e-12), 'online 应等于三遍 stable'
# 对拍超大值行：online 也绝不溢出（全程相对运行 max）
big = np.array([1000.0, 1001.0, 1002.0, 999.0])
ob = online_softmax(big)
print('online(大值):', np.round(ob, 4), ' 无 nan =', not np.isnan(ob).any())
assert np.allclose(ob, softmax_stable(big), atol=1e-12) and not np.isnan(ob).any()
print('✅ online softmax 一遍算完，与三遍 stable 逐位一致，且天然数值稳定')

## 7 · 分块 online softmax：合并块状态（FlashAttention 发动机）⭐

把「每来一个元素更新」升级为「每来一**块**更新」。每块算局部 `(m_block, l_block)`，再用同样的校正因子规则**合并**两个块的状态：

$$m=\max(m_1,m_2),\quad l=l_1 e^{m_1-m}+l_2 e^{m_2-m}$$

这个 merge 是**可结合**的，且最终结果**与块大小无关**——这正是 FlashAttention 能对 K/V 分块的关键。

In [ ]:
def block_state(xb):
    '''一个块的局部 softmax 状态 (max, 相对该 max 的指数和)。'''
    m = xb.max()
    l = np.exp(xb - m).sum()
    return m, l

def merge_states(s1, s2):
    '''合并两个 (m,l) 状态：各自相对全局新 max 缩放再相加。单位元 = (-inf, 0)。'''
    m1, l1 = s1; m2, l2 = s2
    m = max(m1, m2)
    l = l1 * np.exp(m1 - m) + l2 * np.exp(m2 - m)
    return m, l

def online_softmax_blocked(x, block=4):
    state = (-np.inf, 0.0)                 # 单位元
    for i in range(0, len(x), block):
        state = merge_states(state, block_state(x[i:i + block]))
    m, l = state
    return np.exp(x - m) / l

x = rng.standard_normal(60)
ref = softmax_stable(x)
for blk in [1, 4, 7, 16, 60]:              # 块大小无关性
    out = online_softmax_blocked(x, blk)
    assert np.allclose(out, ref, atol=1e-12), f'block={blk} 应与整行一致'
print('✅ 分块 online softmax：块大小 1/4/7/16/60 结果全部一致，且 == 整行 softmax')
print('   这个 (m,l) 状态合并就是 FlashAttention 跨 KV 块的递推骨架（下一模块再加输出累加器 O）')

---
## ✏️ 练习 1：从零实现树形求最大 + 数步数

实现 `tree_reduce_max_counted(x)`，返回 `(最大值, 轮数)`。用「步长减半」模式、`-inf` 作单位元。
验证：对非 2 的幂长度也等于 `np.max`，且轮数 = `ceil(log2(补齐后的size))`。

In [ ]:
def tree_reduce_max_counted(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    # TODO:
    #  1. size = 补到 2 的幂： 1 << (n-1).bit_length()  (n>1)
    #  2. buf 用 -inf 补齐，前 n 个放 x
    #  3. stride 从 size//2 折半到 1，每轮 buf[:stride]=maximum(buf[:stride], buf[stride:2*stride])，计数
    #  4. 返回 (buf[0], rounds)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
import math
for n in [1, 5, 16, 100, 1000]:
    xx = rng.standard_normal(n)
    mx, rounds = tree_reduce_max_counted(xx)
    assert np.isclose(mx, xx.max()), f'n={n} 最大值应匹配 np.max'
    size = 1 << (n - 1).bit_length() if n > 1 else 1
    assert rounds == int(round(math.log2(size))), f'n={n} 轮数应为 log2(size)'
print('✅ 练习 1 通过：树形求最大正确，深度 = log2(补齐 size)')

## ✏️ 练习 2：从零实现 online softmax

实现一遍流式的 `online_softmax_ex(x)`，与三遍 `softmax_stable` 对拍。
关键：运行 `m` 初始化为 `-inf`、`l` 初始化为 `0.0`；每步先算 `m_new`，再用校正因子更新 `l`。

In [ ]:
def online_softmax_ex(x):
    x = np.asarray(x, dtype=float)
    m = -np.inf
    l = 0.0
    # TODO: 遍历 x：m_new=max(m,xi); l = l*exp(m-m_new)+exp(xi-m_new); m=m_new
    #       最后 return exp(x-m)/l
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
for _ in range(5):
    xx = rng.standard_normal(50) * 5      # 拉大尺度
    assert np.allclose(online_softmax_ex(xx), softmax_stable(xx), atol=1e-12)
huge = np.array([5000.0, 5001.0, 4999.0, 5002.0])   # 朴素会溢出
out = online_softmax_ex(huge)
assert not np.isnan(out).any(), 'online 必须不溢出'
assert np.allclose(out, softmax_stable(huge), atol=1e-12)
assert np.isclose(out.sum(), 1.0)
print('✅ 练习 2 通过：一遍 online softmax == 三遍 stable，且大值不溢出')

## ✏️ 练习 3：分块状态合并 + 块大小无关性

实现 `merge_states_ex(s1, s2)` 和 `blocked_ex(x, block)`，验证**任意块大小给出相同结果**，且等于整行 softmax。
这是 FlashAttention 跨 KV 块递推的直接预演。

In [ ]:
def merge_states_ex(s1, s2):
    # s = (m, l). TODO: m=max(m1,m2); l=l1*exp(m1-m)+l2*exp(m2-m); 返回 (m,l)
    raise NotImplementedError

def blocked_ex(x, block):
    x = np.asarray(x, dtype=float)
    state = (-np.inf, 0.0)
    # TODO: 逐块: mb=blk.max(); lb=exp(blk-mb).sum(); state=merge_states_ex(state,(mb,lb))
    #       最后 m,l=state; return exp(x-m)/l
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
x = rng.standard_normal(48) * 3
ref = softmax_stable(x)
outs = [blocked_ex(x, b) for b in [1, 2, 3, 6, 48]]
for b, o in zip([1, 2, 3, 6, 48], outs):
    assert np.allclose(o, ref, atol=1e-12), f'block={b} 应等于整行'
for o in outs[1:]:
    assert np.allclose(o, outs[0], atol=1e-12), '不同块大小应给出相同结果'
# 合并的结合律：(A,B),C 与 A,(B,C) 一致
sa, sb, sc = (0.0, 1.0), (2.0, 3.0), (-1.0, 0.5)
left = merge_states_ex(merge_states_ex(sa, sb), sc)
right = merge_states_ex(sa, merge_states_ex(sb, sc))
assert np.allclose(left, right, atol=1e-12), 'merge 必须满足结合律'
print('✅ 练习 3 通过：状态合并满足结合律，分块结果与块大小无关')

## ✏️ 练习 4：在线 LogSumExp

softmax 的归一化项对数 `LSE(x)=log Σ exp(x_j)`，数值稳定写法是 `m + log(Σ exp(x_j - m))`。
FlashAttention 反向就靠每行存一个 LSE 标量来重算概率。用在线 `(m, l)` 直接得到：`LSE = m + log(l)`。

实现 `logsumexp_online(x)`（复用在线 `(m,l)`），与稳定参考对拍。

In [ ]:
def logsumexp_online(x):
    x = np.asarray(x, dtype=float)
    m = -np.inf; l = 0.0
    # TODO: 同 online_softmax 维护 (m,l)，但最后 return m + log(l)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
def lse_ref(x):
    m = x.max(); return m + np.log(np.exp(x - m).sum())
for _ in range(5):
    xx = rng.standard_normal(40) * 4
    assert np.isclose(logsumexp_online(xx), lse_ref(xx), atol=1e-10)
big = np.array([800.0, 801.0, 802.0])
assert np.isclose(logsumexp_online(big), lse_ref(big), atol=1e-9)
assert not np.isnan(logsumexp_online(big))
# 一致性：softmax(x) 应等于 exp(x - LSE)
xx = rng.standard_normal(30)
assert np.allclose(np.exp(xx - logsumexp_online(xx)), softmax_stable(xx), atol=1e-12)
print('✅ 练习 4 通过：在线 LSE = m + log(l)，且 softmax(x) == exp(x - LSE)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def tree_reduce_max_counted(x):
    x = np.asarray(x, dtype=float); n = len(x)
    size = 1 << (n - 1).bit_length() if n > 1 else 1
    buf = np.full(size, -np.inf); buf[:n] = x
    stride = size // 2; rounds = 0
    while stride >= 1:
        buf[:stride] = np.maximum(buf[:stride], buf[stride:2 * stride])
        stride //= 2; rounds += 1
    return buf[0], rounds

In [ ]:
# 练习 2 参考答案
def online_softmax_ex(x):
    x = np.asarray(x, dtype=float)
    m = -np.inf; l = 0.0
    for xi in x:
        m_new = max(m, xi)
        l = l * np.exp(m - m_new) + np.exp(xi - m_new)
        m = m_new
    return np.exp(x - m) / l

In [ ]:
# 练习 3 参考答案
def merge_states_ex(s1, s2):
    m1, l1 = s1; m2, l2 = s2
    m = max(m1, m2)
    l = l1 * np.exp(m1 - m) + l2 * np.exp(m2 - m)
    return m, l

def blocked_ex(x, block):
    x = np.asarray(x, dtype=float)
    state = (-np.inf, 0.0)
    for i in range(0, len(x), block):
        blk = x[i:i + block]
        state = merge_states_ex(state, (blk.max(), np.exp(blk - blk.max()).sum()))
    m, l = state
    return np.exp(x - m) / l

In [ ]:
# 练习 4 参考答案
def logsumexp_online(x):
    x = np.asarray(x, dtype=float)
    m = -np.inf; l = 0.0
    for xi in x:
        m_new = max(m, xi)
        l = l * np.exp(m - m_new) + np.exp(xi - m_new)
        m = m_new
    return m + np.log(l)

---
## 🧪 真实数据胶囊：一行真实注意力的 softmax 与 HBM 账

用一条**真实尺度**的注意力 logits 行（序列长 `n=2048`、头维 `d=128`、缩放 `1/√d`）验证 online == stable，再算一笔 **HBM 流量账**：朴素稳定 softmax 三遍读整行，融合在线 softmax 一遍读整行——这就是融合内核省下的带宽。

In [ ]:
# 真实尺度的一行注意力分数：q·k / sqrt(d)，n 个 key
n, d = 2048, 128
q = rng.standard_normal(d)
Km = rng.standard_normal((n, d))
logits = (Km @ q) / np.sqrt(d)          # (n,) 真实缩放点积分数
print(f'logits 范围 [{logits.min():.2f}, {logits.max():.2f}], n={n}')

p_stable = softmax_stable(logits)
p_online = online_softmax(logits)
assert np.allclose(p_online, p_stable, atol=1e-12), '真实行上 online == stable'
assert np.isclose(p_stable.sum(), 1.0)
print('✅ 真实注意力行：online softmax 与稳定 softmax 逐位一致')
print(f'   最大注意力权重={p_stable.max():.4f}, 有效关注的 key 数(>1e-3)={int((p_stable>1e-3).sum())}')

**🧪 胶囊练习**：实现 `softmax_passes_bytes(n, passes, b=4)`：一行 `n` 个元素、每遍把整行读一次（`b` 字节/元素），返回 `passes` 遍读取的总字节数。用它对比朴素三遍 vs 融合一遍省下多少 HBM 流量。

In [ ]:
def softmax_passes_bytes(n, passes, b=4):
    # TODO: 返回 passes * n * b （每遍把整行 n 个元素各读一次，每个 b 字节）
    raise NotImplementedError

In [ ]:
# 自测
naive_bytes = softmax_passes_bytes(2048, passes=3)   # 求max + 求sum + 写回前的读
fused_bytes = softmax_passes_bytes(2048, passes=1)   # 融合/在线：整行只读一遍
assert naive_bytes == 3 * 2048 * 4
assert fused_bytes == 2048 * 4
assert naive_bytes == 3 * fused_bytes
print(f'朴素三遍读 = {naive_bytes} B, 融合一遍读 = {fused_bytes} B, 省 {naive_bytes/fused_bytes:.0f}x HBM 读流量')
print('✅ 胶囊练习通过：在线/融合把 softmax 的 HBM 读流量降到 1/3（访存受限算子的直接收益）')

In [ ]:
# 📖 胶囊参考答案
def softmax_passes_bytes(n, passes, b=4):
    return passes * n * b

---
## 🔧 旁注：对应的 Triton 融合 softmax 内核

本模块的「稳定 softmax 一遍读完」在 Triton 里就是经典的 fused-softmax 内核（伪代码，**本环境不跑**）：

```python
import triton, triton.language as tl

@triton.jit
def softmax_kernel(x_ptr, y_ptr, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)                    # 每个 program 处理一行
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=-float('inf'))
    m   = tl.max(x, axis=0)                   # 规约①：行最大（树形，编译器生成）
    num = tl.exp(x - m)                       # 稳定化
    den = tl.sum(num, axis=0)                 # 规约②：归一化和
    tl.store(y_ptr + row * n_cols + cols, num / den, mask=mask)
    # 整行只从 HBM 读一次、中间 exp 留在寄存器 -> 融合，省掉两遍 HBM 往返
```

对应关系：`tl.max`/`tl.sum` ↔ 我们的树形规约；整行驻留寄存器 ↔ 一遍扫描。
当一行**大到放不进一个 program 的寄存器**时（如超长序列），就改用我们第 6/7 节的 **online / 分块** 版本——这正是从 fused-softmax 通向 **FlashAttention** 的桥。

### 小结
- 规约天生串行；用**结合律**重排成 **O(log n)** 深的树（求和/求最大同构）。
- warp 内用 **shuffle** 规约（5 步、无 shared/barrier），warp 间用 shared —— 两级 block 规约。
- 浮点**不结合**：并行与串行规约结果差最后几位是正常的；对拍用 `np.allclose` 不用 `==`（成对求和还更精确）。
- 稳定 softmax 减最大值防溢出，但要**三遍**；**online softmax** 用校正因子 `e^{m-m_new}` 一遍算完。
- **分块 online softmax** 把 `(m,l)` 状态合并，结果与块大小无关 —— 这就是 **FlashAttention 的发动机**。

下一站：**模块 05 · FlashAttention** —— 给 `(m,l)` 状态再加一个输出累加器 `O`，永不物化 n×n 分数矩阵。